In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt

# データの読み込み
# MobileNetV2が推奨する224x224にリサイズ
DATA_DIR = "dog_cat_photos"

train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    f"{DATA_DIR}/train",
    image_size=(224, 224),
    label_mode="binary",
    batch_size=32,
    shuffle=True
)

test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    f"{DATA_DIR}/test",
    image_size=(224, 224),
    label_mode="binary",
    batch_size=32,
    shuffle=False
)

# 画像の水増し関数の定義
def flip_left_right(image, label):
    return tf.image.flip_left_right(image), label

def flip_up_down(image, label):
    return tf.image.flip_up_down(image), label

def rot90(image, label):
    return tf.image.rot90(image), label

# 水増し処理の実行と連結
train_dataset_lr = train_dataset.map(flip_left_right)
train_dataset_ud = train_dataset.map(flip_up_down)
train_dataset_r90 = train_dataset.map(rot90)

# 連結して訓練データへ
train_dataset = train_dataset.concatenate(train_dataset_lr)
train_dataset = train_dataset.concatenate(train_dataset_ud)
train_dataset = train_dataset.concatenate(train_dataset_r90)

# データをシャッフルする
train_dataset = train_dataset.shuffle(32)

# MobileNetV2モデルの作成
# 入力層の定義
input_layer = tf.keras.Input(shape=(224, 224, 3))

# 学習済みモデル（本体）の読み込み
preprocess_layer = tf.keras.applications.mobilenet_v2.preprocess_input(input_layer)

base_model = tf.keras.applications.mobilenet_v2.MobileNetV2(
    input_shape=(224, 224, 3),
    input_tensor=preprocess_layer,
    include_top=False,
    weights="imagenet",
    pooling='avg' 
)

# 重みを固定（転移学習）
base_model.trainable = False

# 最終的なモデルの組み立て
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.Dense(1, activation='sigmoid') # 犬か猫かの2値判定
])

# コンパイルと学習
model.compile(
    optimizer="adam",
    loss='binary_crossentropy',
    metrics=["accuracy"]
)

# 学習開始
print("\n--- 学習開始 ---")
model.fit(train_dataset, epochs=10)

# 最終評価
print("\n--- 最終評価 (evaluate) ---")
loss, accuracy = model.evaluate(test_dataset)

print(f"\nテストデータの正答率: {accuracy * 100:.2f} %")

Found 300 files belonging to 2 classes.
Found 100 files belonging to 2 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

--- 学習開始 ---
Epoch 1/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 12s 227ms/step - accuracy: 0.7733 - loss: 0.4735
Epoch 2/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 9s 228ms/step - accuracy: 0.9217 - loss: 0.2270
Epoch 3/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 9s 228ms/step - accuracy: 0.9383 - loss: 0.1724
Epoch 4/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 230ms/step - accuracy: 0.9533 - loss: 0.1454
Epoch 5/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 9s 227ms/step - accuracy: 0.9642 - loss: 0.1257
Epoch 6/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 9s 226ms/step - accuracy: 0.9700 - loss: 0.1115
Epoch 7/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 9s 225ms/step - accuracy: 0.9742 - loss: 0.1001
Epoch 8/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 9s 227ms/step - accuracy: 0.9792 - loss: 0.0908
Epoch 9/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 9s 225ms/step - accuracy: 0.9800 - loss: 0.0827
Epoch 10/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 230ms/step - accuracy: 0.9808 - loss: 